In [ ]:
#pip install plotly

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   -------------------------------------- - 9.2/9.6 MB 47.7 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 42.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [15]:
# PREAMBLE (run once)
import pandas as pd
import plotly.express as px
import numpy as np
from pathlib import Path

# ---------- paths -----------------------------------------------------------
CLEAN_DIR = Path (r"D:/CIM Warwick/Dissertation/Datasets/All Datasets/clean/")


In [16]:
import os
import re
import pandas as pd

# ── 1. Locate the tidy Parquets ────────────────────────────────────────────────
world_pop_path = os.path.join(CLEAN_DIR, "world_pop_long.parquet")
dc_counts_path = os.path.join(CLEAN_DIR, "global_dc_counts_tidy.parquet")

# ── 2. Load the data (needs pyarrow or fastparquet) ────────────────────────────
# If you haven’t installed either engine yet:
#   pip install pyarrow            # or fastparquet
world_pop = pd.read_parquet(world_pop_path)
dc_counts = pd.read_parquet(dc_counts_path)

# ── 3. Regex that captures common USA spellings / abbreviations ────────────────
us_pattern = re.compile(
    r"\b(?:United\s*States(?:\s*of\s*America)?|USA?|U\.S\.A?\.?)\b",
    flags=re.IGNORECASE,
)

def list_us_variants(df, country_col="country"):
    """Return a sorted list of country-name strings in *df[country_col]*
       that match our USA pattern."""
    uniques = df[country_col].dropna().unique()
    return sorted({name for name in uniques if us_pattern.search(name)})

# ── 4. Inspect the two datasets ────────────────────────────────────────────────
print("🔎 US variants in World Bank population:")
print(list_us_variants(world_pop, "country"), end="\n\n")

print("🔎 US variants in global DC-count data:")
print(list_us_variants(dc_counts, "country"))


🔎 US variants in World Bank population:
['United States']

🔎 US variants in global DC-count data:
['United States']


In [17]:
# Assume world_pop and dc_counts already loaded
rename_map = {
    "United States of America": "United States",
}

dc_counts["country"] = dc_counts["country"].replace(rename_map)

# --- optional: tidy World-Bank variants too
world_pop["country"] = world_pop["country"].str.replace(r"\s*\(US\)$", "", regex=True)


In [18]:
usa_dc = dc_counts[dc_counts['country'].str.contains('United States|USA', case=False, na=False)]
print("USA Data Center count:")
print(usa_dc)

usa_pop = world_pop[world_pop['country'].str.contains('United States', case=False, na=False)]
print("USA Population Data:")
print(usa_pop)

USA Data Center count:
      region        country  dc_count
16  Americas  United States      2052
USA Population Data:
             country iso3  year   population
246    United States  USA  1960  180671000.0
506    United States  USA  1961  183691000.0
766    United States  USA  1962  186538000.0
1026   United States  USA  1963  189242000.0
1286   United States  USA  1964  191889000.0
...              ...  ...   ...          ...
15877  United States  USA  2020  331577720.0
16138  United States  USA  2021  332099760.0
16399  United States  USA  2022  334017321.0
16660  United States  USA  2023  336806231.0
16921  United States  USA  2024  340110988.0

[65 rows x 4 columns]


In [19]:
# Save the updated dataframes back to parquet files
world_pop.to_parquet(world_pop_path)
dc_counts.to_parquet(dc_counts_path)

In [21]:
# ──────────────────────────────────────────────────────────────────────────────
#  EDA scaffold for each tidy dataset in /mnt/data/clean/
# ──────────────────────────────────────────────────────────────────────────────
#
#  What it does:
#    • Loads every *.parquet in CLEAN_DIR, one at a time
#    • Prints basic shape / memory footprint
#    • Shows column dtypes and % missing
#    • Displays five random sample rows
#    • (Optionally) calls quick-look helpers specific to known datasets
#
#  How to extend:
#    • Drop your own plot / describe / groupby calls in the per-dataset helpers
#    • Or wrap the whole loop in tqdm for progress bars
# ──────────────────────────────────────────────────────────────────────────────

from pandas.api.types import is_numeric_dtype

# ---------- generic utilities -------------------------------------------------
def print_banner(title, char="═"):
    print(f"\n{char * 4} {title} {char * (76 - len(title))}")

def dataset_summary(df: pd.DataFrame, name: str):
    """Lightweight textual summary."""
    print_banner(f"{name}: basic info")
    rows, cols = df.shape
    print(f"• rows : {rows:,}\n• cols : {cols}\n• memory: {df.memory_usage(deep=True).sum()/1e6:,.1f} MB")

    # column summary
    print_banner(f"{name}: dtypes & NaNs")
    col_stats = (
        df.isna()
          .mean()
          .mul(100)
          .round(2)
          .rename("pct_na")
          .to_frame()
          .assign(dtype=[df[c].dtype for c in df.columns])
    )
    display(col_stats.head(20))  # first 20 cols for brevity

    # show a tiny random sample
    print_banner(f"{name}: random sample")
    display(df.sample(min(5, len(df)), random_state=42))

# ---------- dataset-specific hooks -------------------------------------------
def quicklook_seds(df):
    """SEDS US energy, price, CO₂ indicators (1960-2023)."""
    print_banner("SEDS quick-look: year span & msn head")
    print(f"year: {df['year'].min()} → {df['year'].max()}")
    print("unique msns (first 10):", ", ".join(df['msn'].unique()[:10]))

def quicklook_world_pop(df):
    """World Bank population tidy."""
    print_banner("World pop quick-look: year span & country count")
    print(f"year: {df['year'].min()} → {df['year'].max()}")
    print(f"countries: {df['country'].nunique()} (iso3 unique: {df['iso3'].nunique() if 'iso3' in df else 'n/a'})")

def quicklook_ukpn(df):
    """UKPN half-hourly utilisation ratios."""
    print_banner("UKPN quick-look: date range & site count")
    print(f"datetime_local: {df['datetime_local'].min()} → {df['datetime_local'].max()}")
    print(f"unique sites: {df['site_id'].nunique()} (types: {df['dctype'].unique()})")

def quicklook_workstation(df):
    """High-freq workstation traces."""
    print_banner("Workstation quick-look: timeframe & cols")
    print(f"datetime: {df['datetime'].min()} → {df['datetime'].max()}")
    numeric_cols = [c for c in df.columns if is_numeric_dtype(df[c])]
    print(f"numeric columns (first 10): {numeric_cols[:10]}")

def quicklook_dc_counts(df):
    """Global data-centre counts."""
    print_banner("Global DC quick-look: by region")
    print(df.groupby('region', dropna=False)['dc_count'].sum().sort_values(ascending=False).head())

# map filename-substring ➜ function
HOOKS = {
    "seds_all": quicklook_seds,
    "world_pop_long": quicklook_world_pop,
    "ukpn_timeseries": quicklook_ukpn,
    "workstation_2021": quicklook_workstation,
    "global_dc_counts": quicklook_dc_counts,
}

# ---------- main loop ---------------------------------------------------------
for fname in sorted(f for f in os.listdir(CLEAN_DIR) if f.endswith(".parquet")):
    full_path = os.path.join(CLEAN_DIR, fname)
    name = fname.replace(".parquet", "")
    df = pd.read_parquet(full_path)

    # generic summary
    dataset_summary(df, name)

    # dataset-specific quick-look (if we have a hook)
    for key, func in HOOKS.items():
        if key in fname:
            func(df)
            break



════ co2_tidy: basic info ════════════════════════════════════════════════════════
• rows : 2,176
• cols : 5
• memory: 0.4 MB

════ co2_tidy: dtypes & NaNs ═════════════════════════════════════════════════════


,pct_na,dtype
state,0.0,object
msn,0.0,object
year,0.0,int64
value,0.0,float64
dataset,0.0,object



════ co2_tidy: random sample ═════════════════════════════════════════════════════


,state,msn,year,value,dataset
282,US,FFACE,1968,1046.2,co2
1161,US,CLCCE,1994,11.2,co2
1138,US,NNACE,1993,34.3,co2
485,US,CLTCE,1974,1193.1,co2
1476,US,FFRCE,2003,385.8,co2



════ complete_tidy: basic info ═══════════════════════════════════════════════════
• rows : 2,366,854
• cols : 5
• memory: 478.1 MB

════ complete_tidy: dtypes & NaNs ════════════════════════════════════════════════


...

,pct_na,dtype
site_id,0.0,object
datetime_local,0.0,"datetime64[ms, UTC]"
utilisation,0.0,float64



════ ukpn_timeseries_tidy: random sample ═════════════════════════════════════════


,site_id,datetime_local,utilisation
78805,Data Centre #24,2024-11-05 08:00:00+00:00,0.1837
806512,Data Centre #40,2024-01-08 22:30:00+00:00,0.0000
155796,Data Centre #8,2024-06-20 20:00:00+00:00,0.0006
276639,Data Centre #29,2024-06-04 05:30:00+00:00,0.1097
858546,Data Centre #14,2024-04-23 11:30:00+00:00,0.3690



════ UKPN quick-look: date range & site count ════════════════════════════════════
datetime_local: 2024-01-01 00:00:00+00:00 → 2024-12-31 23:30:00+00:00


KeyError: 'dctype'

--- Checking for key/column presence ---

Checking DataFrame columns:
world_pop missing columns: None
dc_counts missing columns: None

Checking HOOKS dictionary keys:
Missing hooks: None
